In [ ]:
#!/usr/bin/env python
# coding: utf-8

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, roc_auc_score


df = pd.read_csv("data/diabetes_dataset.csv")
df_model = df.copy()

ord_enc = OrdinalEncoder()
df_model["location"]        = ord_enc.fit_transform(df_model[["location"]])
df_model["smoking_history"] = ord_enc.fit_transform(df_model[["smoking_history"]])
df_model["year"]            = ord_enc.fit_transform(df_model[["year"]])

lb = LabelEncoder()
df_model["gender"] = lb.fit_transform(df_model["gender"])

race_cols = [c for c in df_model.columns if "race:" in c]
df_model.drop(columns=race_cols, inplace=True)

x = df_model.drop("diabetes", axis=1)
y = df_model["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=33, stratify=y)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


MODELS = {
    "Logistic Regression": LogisticRegression(penalty="l2", max_iter=1000),
    "Decision Tree":       DecisionTreeClassifier(),
    "Random Forest":       RandomForestClassifier(n_estimators=20),
    "SVC":                 SVC(),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
    "XGBoost":             XGBClassifier(n_estimators=50, eval_metric="logloss"),
}

results = []

for name, model in MODELS.items():
    use_scaled = name in ["Logistic Regression", "SVC", "KNN"]
    Xtr = X_train_sc if use_scaled else X_train.values
    Xte = X_test_sc  if use_scaled else X_test.values

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)

    results.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, y_pred),  4),
        "F1 Score":  round(f1_score(y_test, y_pred),        4),
        "Recall":    round(recall_score(y_test, y_pred),     4),
        "Precision": round(precision_score(y_test, y_pred),  4),
        "AUC":       round(roc_auc_score(y_test, y_pred),    4),
    })

results_df = pd.DataFrame(results).set_index("Model")
results_df = results_df.sort_values("F1 Score", ascending=False)

print(results_df.to_string())